Utilizando a porta XOR

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# --- Funções de Ativação
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

# --- 1. Definir os dados do XOR
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])

Y = np.array([[0],
              [1],
              [1],
              [0]])

# --- 2. Inicializar a Rede
np.random.seed(42)

input_neurons = 2
hidden_neurons = 2
output_neurons = 1

learning_rate = 0.2
epochs = 2000

weights_hidden = np.random.uniform(size=(input_neurons, hidden_neurons))
bias_hidden = np.random.uniform(size=(1, hidden_neurons))
weights_output = np.random.uniform(size=(hidden_neurons, output_neurons))
bias_output = np.random.uniform(size=(1, output_neurons))

error_history = []
epoch_list = []

for i in range(epochs):

    # --- FORWARD ---
    hidden_layer_input = np.dot(X, weights_hidden) + bias_hidden
    hidden_layer_activation = sigmoid(hidden_layer_input)
    output_layer_input = np.dot(hidden_layer_activation, weights_output) + bias_output
    predicted_output = sigmoid(output_layer_input)

    # --- BACKWARD ---
    error = Y - predicted_output

    delta_output = error * sigmoid_derivative(predicted_output)
    error_hidden_layer = delta_output.dot(weights_output.T)
    delta_hidden = error_hidden_layer * sigmoid_derivative(hidden_layer_activation)

    # --- Atualização de Pesos
    weights_output += hidden_layer_activation.T.dot(delta_output) * learning_rate
    bias_output += np.sum(delta_output, axis=0, keepdims=True) * learning_rate
    weights_hidden += X.T.dot(delta_hidden) * learning_rate
    bias_hidden += np.sum(delta_hidden, axis=0, keepdims=True) * learning_rate

    if (i % 100) == 0:
        current_error = np.mean(np.abs(error))
        error_history.append(current_error)
        epoch_list.append(i)

hidden_layer_input = np.dot(X, weights_hidden) + bias_hidden
hidden_layer_activation = sigmoid(hidden_layer_input)
output_layer_input = np.dot(hidden_layer_activation, weights_output) + bias_output
predicted_output = sigmoid(output_layer_input)


print("\n--- Resultados Pós-Treinamento ---")

#Curva de Aprendizado (Erro vs. Épocas)
print("Exibindo Gráfico 1: Curva de Aprendizado...")
plt.figure(figsize=(10, 5))
plt.plot(epoch_list, error_history)
plt.title('Curva de Aprendizado (Erro ao Longo das Épocas)')
plt.xlabel('Época')
plt.ylabel('Erro Médio Absoluto')
plt.grid(True)
plt.show()


# Tabela de Resultados 
print("\nExibindo Tabela de Resultados:")
data = {
    'Entrada_1': X[:, 0],
    'Entrada_2': X[:, 1],
    'Esperado': Y.flatten(),
    'Previsto (Bruto)': predicted_output.flatten(),
    'Previsão (Final)': np.round(predicted_output.flatten())
}
results_df = pd.DataFrame(data)
results_df['Previsto (Bruto)'] = results_df['Previsto (Bruto)'].map('{:,.4f}'.format)
results_df['Previsão (Final)'] = results_df['Previsão (Final)'].astype(int)
print(results_df.to_string(index=False))


print("\nExibindo Gráfico 2: Fronteira de Decisão...")

def plot_decision_boundary(X, y, wh, bh, wo, bo):
    x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
    y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1

    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                         np.linspace(y_min, y_max, 100))

    grid_data = np.c_[xx.ravel(), yy.ravel()]

    h_input = np.dot(grid_data, wh) + bh
    h_act = sigmoid(h_input)
    o_input = np.dot(h_act, wo) + bo
    o_act = sigmoid(o_input)

    Z = np.round(o_act)
    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, cmap=plt.cm.RdYlBu, alpha=0.6)

    scatter = plt.scatter(X[:, 0], X[:, 1], c=y.flatten(), s=100,
                        edgecolor='k', cmap=plt.cm.RdYlBu)

    plt.title('Fronteira de Decisão da Rede Neural (XOR)')
    plt.xlabel('Entrada 1')
    plt.ylabel('Entrada 2')
    plt.legend(handles=scatter.legend_elements()[0],
               labels=['Classe 0', 'Classe 1'])
    plt.grid(True)
    plt.show()

plot_decision_boundary(X, Y, weights_hidden, bias_hidden, weights_output, bias_output)

Classificação Multiclasse

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix

# --- Funções de Ativação

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    # 1.0 para valores > 0, e 0.0 para valores <= 0
    return (x > 0).astype(float)

def softmax(x):
    exp_values = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_values / np.sum(exp_values, axis=1, keepdims=True)

# --- 1. Dados do Display
X = np.array([
    [1, 1, 1, 1, 1, 1, 0],  # 0
    [0, 1, 1, 0, 0, 0, 0],  # 1
    [1, 1, 0, 1, 1, 0, 1],  # 2
    [1, 1, 1, 1, 0, 0, 1],  # 3
    [0, 1, 1, 0, 0, 1, 1],  # 4
    [1, 0, 1, 1, 0, 1, 1],  # 5
    [1, 0, 1, 1, 1, 1, 1],  # 6
    [1, 1, 1, 0, 0, 0, 0],  # 7
    [1, 1, 1, 1, 1, 1, 1],  # 8
    [1, 1, 1, 1, 0, 1, 1]   # 9
])

Y = np.array([
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0], # 0
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 0], # 1
    [0, 0, 1, 0, 0, 0, 0, 0, 0, 0], # 2
    [0, 0, 0, 1, 0, 0, 0, 0, 0, 0], # 3
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0], # 4
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0], # 5
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0], # 6
    [0, 0, 0, 0, 0, 0, 0, 1, 0, 0], # 7
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 0], # 8
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]  # 9
])

# --- 2. Inicializar a Rede
np.random.seed(42)
input_neurons = 7
hidden_neurons = 12
output_neurons = 10
learning_rate = 0.08
epochs = 100

weights_hidden = np.random.uniform(size=(input_neurons, hidden_neurons))
bias_hidden = np.random.uniform(size=(1, hidden_neurons))
weights_output = np.random.uniform(size=(hidden_neurons, output_neurons))
bias_output = np.random.uniform(size=(1, output_neurons))

loss_history = []
epoch_list = []

for i in range(epochs):

    hidden_layer_input = np.dot(X, weights_hidden) + bias_hidden
    hidden_layer_activation = relu(hidden_layer_input)
    output_layer_input = np.dot(hidden_layer_activation, weights_output) + bias_output
    predicted_probabilities = softmax(output_layer_input)

    delta_output = predicted_probabilities - Y
    error_hidden_layer = delta_output.dot(weights_output.T)
    delta_hidden = error_hidden_layer * relu_derivative(hidden_layer_activation)

    weights_output -= hidden_layer_activation.T.dot(delta_output) * learning_rate
    bias_output -= np.sum(delta_output, axis=0, keepdims=True) * learning_rate
    weights_hidden -= X.T.dot(delta_hidden) * learning_rate
    bias_hidden -= np.sum(delta_hidden, axis=0, keepdims=True) * learning_rate

    if (i % 100) == 0:
        loss = -np.sum(Y * np.log(predicted_probabilities + 1e-9)) / len(Y)
        loss_history.append(loss)
        epoch_list.append(i)

hidden_layer_input = np.dot(X, weights_hidden) + bias_hidden
hidden_layer_activation = relu(hidden_layer_input)
output_layer_input = np.dot(hidden_layer_activation, weights_output) + bias_output
predicted_probabilities = softmax(output_layer_input)

predicted_classes = np.argmax(predicted_probabilities, axis=1)
# Pega o índice da classe real
actual_classes = np.argmax(Y, axis=1)
confidence = np.max(predicted_probabilities, axis=1) * 100


print("\n--- Resultados Pós-Treinamento (Display) ---")

# Curva de Aprendizado
print("Exibindo Gráfico 1: Curva de Aprendizado...")
plt.figure(figsize=(10, 5))
plt.plot(epoch_list, loss_history)
plt.title('Curva de Aprendizado (Custo ao Longo das Épocas)')
plt.xlabel('Época')
plt.ylabel('Custo (Cross-Entropy Loss)')
plt.grid(True)
plt.show()

# Tabela de Resultados
print("\nExibindo Tabela de Resultados:")
data = {
    'Dígito Real': actual_classes,
    'Dígito Previsto': predicted_classes,
    'Confiança (%)': confidence
}
results_df = pd.DataFrame(data)
results_df['Confiança (%)'] = results_df['Confiança (%)'].map('{:,.2f}%'.format)
print(results_df.to_string(index=False))


# Matriz de Confusão
print("\nExibindo Gráfico 2: Matriz de Confusão...")

cm = confusion_matrix(actual_classes, predicted_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title('Matriz de Confusão')
plt.ylabel('Valor Verdadeiro (Real)')
plt.xlabel('Valor Previsto')
plt.show()